# Recommendation System - Combined Embeddings

This notebook combines semantic (BGE-M3) and structural (Node2Vec) embeddings to build a recommendation system for WikiCS articles using cosine similarity.

In [ ]:
import pandas as pd
import numpy as np
import os
import json
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
from IPython.display import display

## 1. Load Embeddings and Data

In [9]:
BGE_M3_PATH = "cap-embeddings/BAAI_bge-m3/master_embeddings.parquet"
NODE2VEC_PATH = "cap-embeddings/node2vec/master_embeddings.parquet"
DATA_PATH = "data/data-embeddings.json"

print("Loading embeddings...")
df_bge = pd.read_parquet(BGE_M3_PATH)
df_n2v = pd.read_parquet(NODE2VEC_PATH)

with open(DATA_PATH, "r") as f:
    data = json.load(f)

nodes = data["nodes"]
# Create ID -> Title mapping
id_to_title = {node["id"]: node["title"] for node in nodes}
title_to_id = {node["title"]: node["id"] for node in nodes}

print(f"BGE-M3 nodes: {len(df_bge)}")
print(f"Node2Vec nodes: {len(df_n2v)}")

Loading embeddings...
BGE-M3 nodes: 11701
Node2Vec nodes: 11701


## 2. Concatenate Embeddings

In [10]:
# Ensure they are sorted by id for easy concatenation or merge
df_bge = df_bge.sort_values("id").reset_index(drop=True)
df_n2v = df_n2v.sort_values("id").reset_index(drop=True)

# Check if IDs match exactly before concatenating directly
if (df_bge["id"] == df_n2v["id"]).all():
    combined_embeddings = []
    for e1, e2 in zip(df_bge["embedding"], df_n2v["embedding"]):
        combined_embeddings.append(np.concatenate([e1, e2]))
    
    df_combined = pd.DataFrame({
        "id": df_bge["id"],
        "embedding": combined_embeddings
    })
    print("Direct concatenation successful.")
else:
    print("IDs do not match exactly, performing inner merge...")
    df_combined = pd.merge(df_bge, df_n2v, on="id", suffixes=("_bge", "_n2v"))
    df_combined["embedding"] = df_combined.apply(lambda r: np.concatenate([r["embedding_bge"], r["embedding_n2v"]]), axis=1)
    df_combined = df_combined[["id", "embedding"]]

print(f"Combined embedding shape: {len(df_combined)} nodes, dim: {len(df_combined['embedding'].iloc[0])}")

Direct concatenation successful.
Combined embedding shape: 11701 nodes, dim: 1152


In [11]:
# Save combined embeddings
OUTPUT_DIR = "cap-embeddings/combined"
os.makedirs(OUTPUT_DIR, exist_ok=True)
df_combined.to_parquet(os.path.join(OUTPUT_DIR, "master_embeddings.parquet"), index=False)
print(f"Combined embeddings saved to {OUTPUT_DIR}")

Combined embeddings saved to cap-embeddings/combined


## 3. Recommendation Pipeline

In [12]:
# Pre-calculate embedding matrix for similarity search
embedding_matrix = np.stack(df_combined["embedding"].values)
node_ids = df_combined["id"].values

In [13]:
def get_recommendations_by_title(title, top_k=10, return_df=False):
    if title not in title_to_id:
        return None
    
    node_id = title_to_id[title]
    # Get index of the node in our matrix
    idx = df_combined[df_combined["id"] == node_id].index[0]
    
    # Compute similarity
    query_vec = embedding_matrix[idx].reshape(1, -1)
    similarities = cosine_similarity(query_vec, embedding_matrix)[0]
    
    # Get top k+1 (including query node)
    top_indices = np.argsort(similarities)[::-1][:top_k+1]
    
    results = []
    for i in top_indices:
        sim_id = node_ids[i]
        if sim_id == node_id: continue # skip self
        results.append({
            "title": id_to_title[sim_id],
            "similarity": similarities[i]
        })
    
    res_list = results[:top_k]
    if return_df:
        return pd.DataFrame(res_list)
    return res_list

# Example
test_title = list(title_to_id.keys())[0]
print(f"Recommendations for '{test_title}':")
display(get_recommendations_by_title(test_title, return_df=True))

Recommendations for 'Twilio':


,title,similarity
0,Unified_communications_as_a_service,0.539612
1,Salesforce_Tower_(Indianapolis),0.515155
2,Optimizely,0.484305
3,Xactly_Corporation,0.468475
4,Virtustream,0.465557
5,Marc_Benioff,0.448092
6,DocuWare,0.445144
7,SnapLogic,0.441534
8,Ping_Identity,0.438111
9,Informatica,0.437286


## 4. Get Top 10 for X Articles (Single Table)

In [ ]:
def get_top_10_for_articles(titles):
    rows = []
    for title in titles:
        recs = get_recommendations_by_title(title, top_k=10)
        if recs:
            row = {"Query Article": title}
            for i, res in enumerate(recs):
                row[f"Top {i+1}"] = res["title"]
            rows.append(row)
        else:
            print(f"Warning: Article '{title}' not found.")
    
    return pd.DataFrame(rows)

# Test with a few articles
sample_titles = list(title_to_id.keys())[:20]
df_batch = get_top_10_for_articles(sample_titles)
display(df_batch)

,Query Article,Top 1,Top 2,Top 3,Top 4,Top 5,Top 6,Top 7,Top 8,Top 9,Top 10
0,Twilio,Unified_communications_as_a_service,Salesforce_Tower_(Indianapolis),Optimizely,Xactly_Corporation,Virtustream,Marc_Benioff,DocuWare,SnapLogic,Ping_Identity,Informatica
1,Program_compatibility_date_range,Processor_Control_Region,XOFTspy_Portable_Anti-Spyware,GDocsDrive,StegAlyzerAS,Spam_Reader,ZAP_File,ComputerSupport.com,MetaSAN,OpenFIRST,Team8
2,SYSTAT_(DEC),Real-Time_Multiprogramming_Operating_System,Universal_Time-Sharing_System,NewDos/80,Multi-user_BASIC,Commercial_Operating_System_(COS),STXIT,Business_Operating_System_(software),MS/8,MacJournal,4K_Disk_Monitor_System
3,List_of_column-oriented_DBMSes,SAP_HANA,List_of_in-memory_databases,Oracle_RAC,In-memory_database,SAP_IQ,Clustrix,Column-oriented_DBMS,Redis_Labs,Amazon_Aurora,Comparison_of_object-relational_database_manag...
4,Stealth_wallpaper,Wireless_intrusion_prevention_system,MAC_flooding,Warchalking,KARMA_attack,Network_encryption_cracking,Retina-X_Studios,Rogue_access_point,Columbitech,Wi-Fi_deauthentication_attack,GingerMaster
5,Scalable_TCP,TCP-Illinois,HSTCP,TCP_tuning,Delay-gradient_congestion_control,Zeta-TCP,Sequenced_Packet_Exchange,Load-balanced_switch,Compound_TCP,Reliable_Event_Logging_Protocol,TCP_global_synchronization
6,Carrier_IQ,UIQ,List_of_open-source_mobile_phones,Rooting_(Android),Series_30,S60_(software_platform),Mobile_cloud_computing,Electronic_waste,Series_30+,EPOC_(operating_system),Nokia_Eseries
7,ACF2,Resource_Access_Control_Facility,Password_Authentication_Protocol,OAuth,Woo–Lam,CRAM-MD5,Generic_Security_Services_Application_Program_...,TACACS,Kerberos_(protocol),BSD_Authentication,Pluggable_authentication_module
8,Dorkbot_(malware),United_States_Computer_Emergency_Readiness_Team,HackTool.Win32.HackAV,CERT_C_Coding_Standard,National_Cybersecurity_and_Communications_Inte...,Common_Vulnerability_Scoring_System,National_Infrastructure_Security_Co-ordination...,Einstein_(US-CERT_program),Common_Weakness_Enumeration,National_Cybersecurity_Center_of_Excellence,National_Vulnerability_Database
9,Lout_(software),RTML,ConTeXt,Microsoft_Assistance_Markup_Language,Charity_(programming_language),LysKOM,Troff,Harlequin_RIP,Extensible_Application_Markup_Language,AmigaGuide,ICI_(programming_language)


: 

: 